In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!git clone https://github.com/ultralytics/ultralytics.git
%cd ultralytics
!pip install -e .

Cloning into 'ultralytics'...
remote: Enumerating objects: 109757, done.
remote: Counting objects: 100% (489/489), done.
remote: Compressing objects: 100% (283/283), done.
remote: Total 109757 (delta 307), reused 262 (delta 206), pack-reused 109268 (from 3)
Receiving objects: 100% (109757/109757), 57.72 MiB | 23.35 MiB/s, done.
Resolving deltas: 100% (82494/82494), done.
/kaggle/working/ultralytics
Obtaining file:///kaggle/working/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.4 MB/s eta 0:00:00
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.4.75-0.editable-py3-none-any.whl size=23420 sha256=9706ec75585bfc618df989202e79fb080c63f6ee04ada76ec4cd3506b2366de4
  Stored in directory: /tmp/pip-e

In [3]:
from ultralytics import YOLO

print("Ultralytics Loaded Successfully")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics Loaded Successfully


In [4]:
%%writefile ultralytics/nn/modules/dsconv.py

import torch
import torch.nn as nn

class DSConv(nn.Module):

    def __init__(self, c1, c2, k=3, s=1):
        super().__init__()

        self.offset_conv = nn.Conv2d(
            c1,
            2 * k * k,
            kernel_size=3,
            padding=1
        )

        self.conv = nn.Conv2d(
            c1,
            c2,
            kernel_size=k,
            stride=s,
            padding=k // 2,
            bias=False
        )

        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()

    def forward(self, x):

        offset = self.offset_conv(x)

        x = self.conv(x)

        return self.act(self.bn(x))

Writing ultralytics/nn/modules/dsconv.py


In [5]:
%%writefile ultralytics/nn/modules/dsc_block.py

import torch
import torch.nn as nn

from ultralytics.nn.modules.dsconv import DSConv

class DSCBottleneck(nn.Module):

    def __init__(self, c1, c2, shortcut=True):
        super().__init__()

        self.cv1 = DSConv(c1, c2)

        self.cv2 = DSConv(c2, c2)

        self.add = shortcut and c1 == c2

    def forward(self, x):

        y = self.cv2(self.cv1(x))

        if self.add:
            return x + y

        return y

Writing ultralytics/nn/modules/dsc_block.py


In [6]:
%%writefile ultralytics/nn/modules/dsc_c3k2.py

import torch
import torch.nn as nn

from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.modules.dsc_block import DSCBottleneck

class DSC_C3k2(nn.Module):

    def __init__(self, c1, c2, n=2):

        super().__init__()

        self.cv1 = Conv(c1, c2, 1, 1)

        self.blocks = nn.Sequential(
            *[
                DSCBottleneck(c2, c2)
                for _ in range(n)
            ]
        )

        self.cv2 = Conv(c2, c2, 1, 1)

    def forward(self, x):

        x = self.cv1(x)

        x = self.blocks(x)

        x = self.cv2(x)

        return x

Writing ultralytics/nn/modules/dsc_c3k2.py


In [7]:
path = "ultralytics/nn/modules/__init__.py"

with open(path, "r") as f:
    text = f.read()

if "from .dsconv import DSConv" not in text:
    text = text.replace(
        "from .head import (",
        "from .dsconv import DSConv\n"
        "from .dsc_c3k2 import DSC_C3k2\n\n"
        "from .head import ("
    )

if '"DSConv",' not in text:
    text = text.replace(
        '"SpatialAttention",',
        '"SpatialAttention",\n'
        '    "DSConv",\n'
        '    "DSC_C3k2",'
    )

with open(path, "w") as f:
    f.write(text)

print("Modules Registered")

Modules Registered


In [8]:
from ultralytics.nn.modules.dsconv import DSConv
from ultralytics.nn.modules.dsc_c3k2 import DSC_C3k2

print("Imports Successful")

Imports Successful


In [9]:
path = "ultralytics/nn/tasks.py"

with open(path, "r") as f:
    text = f.read()

if "DSC_C3k2" not in text:

    text = text.replace(
        "v10Detect,",
        "v10Detect,\n"
        "    DSConv,\n"
        "    DSC_C3k2,"
    )

with open(path, "w") as f:
    f.write(text)

print("Import Added")

Import Added


In [10]:
!grep -n "DSC_C3k2" ultralytics/nn/tasks.py

77:    DSC_C3k2,


In [11]:
!grep -n "base_modules" ultralytics/nn/tasks.py

1802:    base_modules = frozenset(
1875:        if m in base_modules:


In [12]:
path = "ultralytics/nn/tasks.py"

with open(path, "r") as f:
    text = f.read()

text = text.replace(
    "A2C2f,\n        }",
    "A2C2f,\n"
    "            DSConv,\n"
    "            DSC_C3k2,\n"
    "        }"
)

with open(path, "w") as f:
    f.write(text)

print("Added To base_modules")

Added To base_modules


In [13]:
!grep -n "if m in base_modules" ultralytics/nn/tasks.py

1879:        if m in base_modules:


In [14]:
path = "ultralytics/nn/tasks.py"

with open(path, "r") as f:
    text = f.read()

old = """        if m in base_modules:
            c1, c2 = ch[f], args[0]"""

new = """        if m is DSC_C3k2:

            c1 = ch[f]
            c2 = args[0]

            args = [c1, c2]

        elif m is DSConv:

            c1 = ch[f]
            c2 = args[0]

            args = [c1, c2]

        elif m in base_modules:
            c1, c2 = ch[f], args[0]"""

text = text.replace(old, new)

with open(path, "w") as f:
    f.write(text)

print("Parser Patched")

Parser Patched


In [15]:
import sys

for mod in list(sys.modules):
    if mod.startswith("ultralytics"):
        del sys.modules[mod]

print("Cache Cleared")

Cache Cleared


In [16]:
%%writefile yolov11_dsc.yaml

nc: 9

scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]

  - [-1, 3, DSC_C3k2, [256]]

  - [-1, 1, Conv, [256, 3, 2]]

  - [-1, 6, DSC_C3k2, [512]]

  - [-1, 1, Conv, [512, 3, 2]]

  - [-1, 6, DSC_C3k2, [512]]

  - [-1, 1, Conv, [1024, 3, 2]]

  - [-1, 3, DSC_C3k2, [1024]]

  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 8], 1, Concat, [1]]
  - [-1, 3, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 3, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C3k2, [1024, True]]

  - [[18, 21, 24], 1, Detect, [nc]]

Writing yolov11_dsc.yaml


In [17]:
from ultralytics import YOLO

model = YOLO("yolo11n.yaml")

print(model.model.yaml)

{'nc': 80, 'scales': {'n': [0.5, 0.25, 1024], 's': [0.5, 0.5, 1024], 'm': [0.5, 1.0, 512], 'l': [1.0, 1.0, 512], 'x': [1.0, 1.5, 512]}, 'backbone': [[-1, 1, 'Conv', [64, 3, 2]], [-1, 1, 'Conv', [128, 3, 2]], [-1, 2, 'C3k2', [256, False, 0.25]], [-1, 1, 'Conv', [256, 3, 2]], [-1, 2, 'C3k2', [512, False, 0.25]], [-1, 1, 'Conv', [512, 3, 2]], [-1, 2, 'C3k2', [512, True]], [-1, 1, 'Conv', [1024, 3, 2]], [-1, 2, 'C3k2', [1024, True]], [-1, 1, 'SPPF', [1024, 5]], [-1, 2, 'C2PSA', [1024]]], 'head': [[-1, 1, 'nn.Upsample', ['None', 2, 'nearest']], [[-1, 6], 1, 'Concat', [1]], [-1, 2, 'C3k2', [512, False]], [-1, 1, 'nn.Upsample', ['None', 2, 'nearest']], [[-1, 4], 1, 'Concat', [1]], [-1, 2, 'C3k2', [256, False]], [-1, 1, 'Conv', [256, 3, 2]], [[-1, 13], 1, 'Concat', [1]], [-1, 2, 'C3k2', [512, False]], [-1, 1, 'Conv', [512, 3, 2]], [[-1, 10], 1, 'Concat', [1]], [-1, 2, 'C3k2', [1024, True]], [[16, 19, 22], 1, 'Detect', ['nc']]], 'scale': 'n', 'yaml_file': 'yolo11n.yaml', 'channels': 3}


In [18]:
%%writefile yolov11_dsc.yaml

nc: 9

scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]

  - [-1, 2, DSC_C3k2, [256]]

  - [-1, 1, Conv, [256, 3, 2]]

  - [-1, 2, DSC_C3k2, [512]]

  - [-1, 1, Conv, [512, 3, 2]]

  - [-1, 2, DSC_C3k2, [512]]

  - [-1, 1, Conv, [1024, 3, 2]]

  - [-1, 2, DSC_C3k2, [1024]]

  - [-1, 1, SPPF, [1024, 5]]

  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]

  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]

  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]

  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]

  - [-1, 2, C3k2, [1024, True]]

  - [[16, 19, 22], 1, Detect, [nc]]

Overwriting yolov11_dsc.yaml


In [19]:
from ultralytics import YOLO

model = YOLO("yolov11_dsc.yaml")

print("Loaded Successfully")

WARNING ⚠️ no model scale passed. Assuming scale='n'.
WARNING ⚠️ WARNING ⚠️ no model scale passed. Assuming scale='n'.
Loaded Successfully


In [20]:
model.info()

YOLOv11_dsc summary: 210 layers, 67,033,019 parameters, 67,033,003 gradients, 336.4 GFLOPs
YOLOv11_dsc summary: 210 layers, 67,033,019 parameters, 67,033,003 gradients, 336.4 GFLOPs


(210, 67033019, 67033003, 336.4168192)

In [21]:
for i, layer in enumerate(model.model.model):
    print(i, type(layer).__name__)

0 Conv
1 Conv
2 DSC_C3k2
3 Conv
4 DSC_C3k2
5 Conv
6 DSC_C3k2
7 Conv
8 DSC_C3k2
9 SPPF
10 C2PSA
11 Upsample
12 Concat
13 C3k2
14 Upsample
15 Concat
16 C3k2
17 Conv
18 Concat
19 C3k2
20 Conv
21 Concat
22 C3k2
23 Detect


In [22]:
for i, layer in enumerate(model.model.model[:11]):
    print("\nLayer", i)
    print(layer)


Layer 0
Conv(
  (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)

Layer 1
Conv(
  (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)

Layer 2
DSC_C3k2(
  (cv1): Conv(
    (conv): Conv2d(32, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): DSCBottleneck(
      (cv1): DSConv(
        (offset_conv): Conv2d(256, 18, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=Tru

In [23]:
%%writefile ultralytics/nn/modules/dsc_c3k2.py

import torch
import torch.nn as nn

from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.modules.dsconv import DSConv


class DSCBottleneck(nn.Module):

    def __init__(self, c, shortcut=True):
        super().__init__()

        self.cv1 = DSConv(c, c, 3)
        self.cv2 = DSConv(c, c, 3)

        self.add = shortcut

    def forward(self, x):

        y = self.cv2(self.cv1(x))

        if self.add:
            return x + y

        return y


class DSC_C3k2(nn.Module):

    def __init__(
        self,
        c1,
        c2,
        shortcut=False,
        e=0.25
    ):
        super().__init__()

        hidden = int(c2 * e)

        self.cv1 = Conv(c1, hidden, 1, 1)

        self.m = nn.Sequential(
            DSCBottleneck(hidden, shortcut),
            DSCBottleneck(hidden, shortcut)
        )

        self.cv2 = Conv(hidden, c2, 1, 1)

    def forward(self, x):

        x = self.cv1(x)

        x = self.m(x)

        x = self.cv2(x)

        return x

Overwriting ultralytics/nn/modules/dsc_c3k2.py


In [24]:
path = "ultralytics/nn/tasks.py"

with open(path, "r") as f:
    text = f.read()

old = """        if m is DSC_C3k2:

            c1 = ch[f]
            c2 = args[0]

            args = [c1, c2]"""

new = """        if m is DSC_C3k2:

            c1 = ch[f]
            c2 = args[0]

            shortcut = args[1] if len(args) > 1 else False
            e = args[2] if len(args) > 2 else 0.25

            args = [c1, c2, shortcut, e]"""

text = text.replace(old, new)

with open(path, "w") as f:
    f.write(text)

print("DSC parser updated")

DSC parser updated


In [25]:
import sys

for mod in list(sys.modules):
    if mod.startswith("ultralytics"):
        del sys.modules[mod]

print("Cache cleared")

Cache cleared


In [26]:
from ultralytics import YOLO

model = YOLO("yolov11_dsc.yaml")

model.info()

WARNING ⚠️ no model scale passed. Assuming scale='n'.
WARNING ⚠️ WARNING ⚠️ no model scale passed. Assuming scale='n'.
WARNING ⚠️ WARNING ⚠️ WARNING ⚠️ no model scale passed. Assuming scale='n'.
YOLOv11_dsc summary: 210 layers, 9,094,715 parameters, 9,094,699 gradients, 35.0 GFLOPs
YOLOv11_dsc summary: 210 layers, 9,094,715 parameters, 9,094,699 gradients, 35.0 GFLOPs
YOLOv11_dsc summary: 210 layers, 9,094,715 parameters, 9,094,699 gradients, 35.0 GFLOPs


(210, 9094715, 9094699, 34.984806400000004)

In [27]:
for i, layer in enumerate(model.model.model[:11]):
    print(i, layer)

0 Conv(
  (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)
1 Conv(
  (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
  (act): SiLU(inplace=True)
)
2 DSC_C3k2(
  (cv1): Conv(
    (conv): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (m): Sequential(
    (0): DSCBottleneck(
      (cv1): DSConv(
        (offset_conv): Conv2d(64, 18, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
   

In [28]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".yaml"):
            print(os.path.join(root, file))

In [29]:
with open("/kaggle/input/datasets/jasonroggy/grazpedwri-dx/folder_structure/yolov5/meta.yaml", "r") as f:
    print(f.read())

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/jasonroggy/grazpedwri-dx/folder_structure/yolov5/meta.yaml'

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/jasonroggy/grazpedwri-dx"):
    if "images" in dirs or "labels" in dirs:
        print(root)

In [ ]:
for root, dirs, files in os.walk("/kaggle/input/datasets/jasonroggy/grazpedwri-dx"):
    if len(files) > 0:
        print(root)
        break

In [ ]:
import os

base = "/kaggle/input/datasets/jasonroggy/grazpedwri-dx/folder_structure/yolov5"

for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    if level >= 2:
        continue

In [ ]:
import os

base = "/kaggle/input/datasets/jasonroggy/grazpedwri-dx/folder_structure/yolov5"

print(os.listdir(base))

In [ ]:
print(os.listdir(
    "/kaggle/input/datasets/jasonroggy/grazpedwri-dx/folder_structure/yolov5/labels"
))

In [ ]:
!find /kaggle/input/datasets/jasonroggy/grazpedwri-dx -type f | head -50

In [ ]:
!find /kaggle/input/datasets/jasonroggy/grazpedwri-dx -type f | head -100

In [ ]:
!find /kaggle/input/datasets/jasonroggy/grazpedwri-dx/folder_structure/yolov5/labels -maxdepth 2 -type d

In [ ]:
import os

labels_path = "/kaggle/input/datasets/jasonroggy/grazpedwri-dx/folder_structure/yolov5/labels"

print("Number of files:", len(os.listdir(labels_path)))

print("\nFirst 20 files:")
for f in sorted(os.listdir(labels_path))[:20]:
    print(f)

In [ ]:
label_files = sorted(os.listdir(labels_path))

print("First label file:")
print(label_files[0])

In [ ]:
with open(os.path.join(labels_path, label_files[0]), "r") as f:
    print(f.read())

In [ ]:
import os

base = "/kaggle/input/datasets/jasonroggy/grazpedwri-dx"

for item in sorted(os.listdir(base)):
    print(item)

In [ ]:
from pathlib import Path

base = Path("/kaggle/input/datasets/jasonroggy/grazpedwri-dx")

image_paths = {}

for png in base.rglob("*.png"):
    image_paths[png.stem] = str(png)

print("Images found:", len(image_paths))

In [ ]:
from pathlib import Path

labels_path = Path(
    "/kaggle/input/datasets/jasonroggy/grazpedwri-dx/folder_structure/yolov5/labels"
)

label_files = list(labels_path.glob("*.txt"))

matches = 0

for lbl in label_files[:100]:
    if lbl.stem in image_paths:
        matches += 1

print("Matches:", matches)

In [ ]:
from pathlib import Path
import random
import os

random.seed(42)

base = Path("/kaggle/input/datasets/jasonroggy/grazpedwri-dx")

labels_dir = base / "folder_structure" / "yolov5" / "labels"

image_paths = {}

for png in base.rglob("*.png"):
    image_paths[png.stem] = str(png)

all_files = []

for lbl in labels_dir.glob("*.txt"):
    stem = lbl.stem

    if stem in image_paths:
        all_files.append(stem)

print("Matched pairs:", len(all_files))

random.shuffle(all_files)

n = len(all_files)

train_split = int(0.70 * n)
val_split = int(0.85 * n)

train_files = all_files[:train_split]
val_files = all_files[train_split:val_split]
test_files = all_files[val_split:]

print("Train:", len(train_files))
print("Val:", len(val_files))
print("Test:", len(test_files))

In [ ]:
from pathlib import Path

work = Path("/kaggle/working/graz_yolo")

for split in ["train", "val", "test"]:

    (work / "images" / split).mkdir(parents=True, exist_ok=True)
    (work / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Folders created")

In [ ]:
import os
from pathlib import Path

labels_dir = Path(
    "/kaggle/input/datasets/jasonroggy/grazpedwri-dx/folder_structure/yolov5/labels"
)

splits = {
    "train": train_files,
    "val": val_files,
    "test": test_files
}

for split, files in splits.items():

    for stem in files:

        img_src = image_paths[stem]
        lbl_src = labels_dir / f"{stem}.txt"

        img_dst = work / "images" / split / f"{stem}.png"
        lbl_dst = work / "labels" / split / f"{stem}.txt"

        if not img_dst.exists():
            os.symlink(img_src, img_dst)

        if not lbl_dst.exists():
            os.symlink(lbl_src, lbl_dst)

print("Symlinks created")

In [ ]:
%%writefile data.yaml

path: /kaggle/working/graz_yolo

train: images/train
val: images/val
test: images/test

nc: 9

names:
  - boneanomaly
  - bonelesion
  - foreignbody
  - fracture
  - metal
  - periostealreaction
  - pronatorsign
  - softtissue
  - text

In [ ]:
from ultralytics.data.utils import check_det_dataset

check_det_dataset("data.yaml")

print("Dataset Valid")

In [ ]:
results = model.train(
    data="data.yaml",
    epochs=1,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    pretrained=False,
    deterministic=False,
    project="/kaggle/working",
    name="YOLO11_DSC_SMOKE"
)

In [ ]:
%%writefile ultralytics/nn/modules/rescbam.py

import torch
import torch.nn as nn


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))

        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=7,
            padding=3,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = torch.mean(x, dim=1, keepdim=True)

        max_out, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        x = torch.cat(
            [avg_out, max_out],
            dim=1
        )

        return self.sigmoid(self.conv(x))


class ResCBAM(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()

    def forward(self, x):

        residual = x

        x = x * self.ca(x)

        x = x * self.sa(x)

        return x + residual

In [ ]:
path = "ultralytics/nn/modules/__init__.py"

In [ ]:
%%writefile ultralytics/nn/modules/rescbam.py

import torch
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=7,
            padding=3,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)

        x = torch.cat([avg_out, max_out], dim=1)

        return self.sigmoid(self.conv(x))


class ResCBAM(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()

    def forward(self, x):

        residual = x

        x = x * self.ca(x)
        x = x * self.sa(x)

        return x + residual

In [ ]:
path = "ultralytics/nn/modules/__init__.py"

with open(path, "r") as f:
    text = f.read()

if "from .rescbam import ResCBAM" not in text:
    text = "from .rescbam import ResCBAM\n" + text

if "ResCBAM" not in text:
    text = text.replace(
        "__all__ = (",
        "__all__ = (\n    'ResCBAM',"
    )

with open(path, "w") as f:
    f.write(text)

print("ResCBAM added to __init__.py")

In [ ]:
import sys

for mod in list(sys.modules):
    if mod.startswith("ultralytics"):
        del sys.modules[mod]

from ultralytics.nn.modules import ResCBAM

print("ResCBAM Import Successful")

In [ ]:
path = "ultralytics/nn/tasks.py"

with open(path, "r") as f:
    text = f.read()

if "ResCBAM" not in text:

    text = text.replace(
        "from ultralytics.nn.modules import (",
        "from ultralytics.nn.modules import (\n    ResCBAM,"
    )

with open(path, "w") as f:
    f.write(text)

print("ResCBAM imported into tasks.py")

In [ ]:
path = "ultralytics/nn/tasks.py"

with open(path, "r") as f:
    text = f.read()

target = "base_modules = frozenset({"

idx = text.find(target)

if idx != -1:

    insert_pos = text.find("{", idx) + 1

    text = (
        text[:insert_pos]
        + "\n        ResCBAM,"
        + text[insert_pos:]
    )

with open(path, "w") as f:
    f.write(text)

print("ResCBAM added to base_modules")

In [ ]:
path = "ultralytics/nn/tasks.py"

with open(path, "r") as f:
    text = f.read()

old = """elif m in base_modules:"""

new = """
elif m is ResCBAM:

            c1 = ch[f]

            args = [c1]

elif m in base_modules:
"""

text = text.replace(old, new)

with open(path, "w") as f:
    f.write(text)

print("Parser updated")

In [ ]:
with open("ultralytics/nn/tasks.py", "r") as f:
    text = f.read()

print("ResCBAM occurrences:", text.count("ResCBAM"))

In [ ]:
with open("ultralytics/nn/tasks.py", "r") as f:
    text = f.read()

for line in text.split("\n"):
    if "ResCBAM" in line:
        print(line)

In [ ]:
with open("ultralytics/nn/tasks.py", "r") as f:
    text = f.read()

idx = text.find("elif m in base_modules")

print(text[idx-400:idx+400])

In [ ]:
with open("ultralytics/nn/tasks.py", "r") as f:
    lines = f.readlines()

for i in range(1875, 1905):
    print(f"{i}: {lines[i]}", end="")

In [ ]:
path = "ultralytics/nn/tasks.py"

with open(path, "r") as f:
    text = f.read()

text = text.replace(
"""elif m is ResCBAM:""",
"""        elif m is ResCBAM:"""
)

text = text.replace(
"""elif m in base_modules:""",
"""        elif m in base_modules:"""
)

with open(path, "w") as f:
    f.write(text)

print("Indentation Fixed")

In [ ]:
with open("ultralytics/nn/tasks.py", "r") as f:
    lines = f.readlines()

for i in range(1880, 1898):
    print(f"{i}: {lines[i]}", end="")

In [ ]:
import sys

for mod in list(sys.modules):
    if mod.startswith("ultralytics"):
        del sys.modules[mod]

from ultralytics.nn.modules import ResCBAM

print("ResCBAM Imported Successfully")

In [ ]:
%%writefile yolov11_dsc_rescbam.yaml

nc: 9

scales:
  n: [0.50, 0.25, 1024]

backbone:

  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]

  - [-1, 2, DSC_C3k2, [256, False, 0.25]]
  - [-1, 1, ResCBAM, []]

  - [-1, 1, Conv, [256, 3, 2]]

  - [-1, 2, DSC_C3k2, [512, False, 0.25]]
  - [-1, 1, ResCBAM, []]

  - [-1, 1, Conv, [512, 3, 2]]

  - [-1, 2, DSC_C3k2, [512, True, 0.25]]
  - [-1, 1, ResCBAM, []]

  - [-1, 1, Conv, [1024, 3, 2]]

  - [-1, 2, DSC_C3k2, [1024, True, 0.25]]
  - [-1, 1, ResCBAM, []]

  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 18], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 14], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

  - [[21, 24, 27], 1, Detect, [nc]]

In [ ]:
with open("yolov11_dsc.yaml", "r") as f:
    print(f.read())

In [ ]:
%%writefile yolov11_dsc_rescbam.yaml

nc: 9

scale: n

scales:
  n: [0.50, 0.25, 1024]

backbone:

  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]

  - [-1, 2, DSC_C3k2, [256, False, 0.25]]
  - [-1, 1, ResCBAM, []]

  - [-1, 1, Conv, [256, 3, 2]]

  - [-1, 2, DSC_C3k2, [512, False, 0.25]]
  - [-1, 1, ResCBAM, []]

  - [-1, 1, Conv, [512, 3, 2]]

  - [-1, 2, DSC_C3k2, [512, True, 0.25]]
  - [-1, 1, ResCBAM, []]

  - [-1, 1, Conv, [1024, 3, 2]]

  - [-1, 2, DSC_C3k2, [1024, True, 0.25]]
  - [-1, 1, ResCBAM, []]

  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 8], 1, Concat, [1]]

  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 5], 1, Concat, [1]]

  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 17], 1, Concat, [1]]

  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 14], 1, Concat, [1]]

  - [-1, 2, C3k2, [1024, True]]

  - [[20, 23, 26], 1, Detect, [nc]]

In [ ]:
import sys

for mod in list(sys.modules):
    if mod.startswith("ultralytics"):
        del sys.modules[mod]

print("Cache Cleared")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov11_dsc_rescbam.yaml")

print("MODEL BUILT SUCCESSFULLY")

In [ ]:
model.info()

In [ ]:
from pathlib import Path

backup_dir = Path("/kaggle/working/YOLO11_DSC_RESCBAM_BACKUP")
backup_dir.mkdir(exist_ok=True)

print("Backup folder created:", backup_dir)

In [ ]:
import shutil
from pathlib import Path

backup_dir = Path("/kaggle/working/YOLO11_DSC_RESCBAM_BACKUP")

files_to_backup = [
    "yolov11_dsc.yaml",
    "yolov11_dsc_rescbam.yaml",
    "data.yaml",
    "ultralytics/nn/modules/rescbam.py",
    "ultralytics/nn/modules/dsconv.py",
    "ultralytics/nn/modules/dsc_c3k2.py",
    "ultralytics/nn/tasks.py",
    "ultralytics/nn/modules/__init__.py"
]

for file in files_to_backup:
    try:
        shutil.copy(file, backup_dir)
        print(f"✓ Saved: {file}")
    except Exception as e:
        print(f"✗ Missing: {file}")

In [ ]:
import os

print("Files backed up:\n")

for f in os.listdir("/kaggle/working/YOLO11_DSC_RESCBAM_BACKUP"):
    print(f)

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/YOLO11_DSC_RESCBAM_BACKUP",
    "zip",
    "/kaggle/working/YOLO11_DSC_RESCBAM_BACKUP"
)

print("ZIP created successfully")

In [ ]:
import os

print(
    os.path.exists(
        "/kaggle/working/YOLO11_DSC_RESCBAM_BACKUP.zip"
    )
)

In [ ]:
results = model.train(
    data="data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,

    pretrained=False,
    deterministic=False,

    optimizer="auto",

    project="/kaggle/working",
    name="YOLO11_DSC_RESCBAM_100E"
)